# 12 — Hybrid CNN + ViT trên MoE-D (Colab A100)

Chạy `experiments/stage10_hybrid.yaml` — **2 ViT × 2 CNN = 4 run, fold 0**, cùng recipe mạnh nhất **M0**
(MoE hướng D · `d_model 256` · overlay `r5_nobce` · clin + derm + metadata · 384 px · 20 epoch, patience 5 · batch hiệu dụng 64).
Mốc so sánh: **M0 fold 0** = Swin-B-384 MoE-D, val Dice **0.601** (đọc từ Drive stage 6, *không train lại*).

| # | Run (`<vit>__abl_<cnn>__f0`) | Nhánh ViT | Nhánh CNN |
|---|---|---|---|
| 1 | `swin_base_384__abl_HC__f0` | Swin-B/384 (IN-22k→1k) | ConvNeXt-B/384 (IN-22k→1k) |
| 2 | `swin_base_384__abl_HE__f0` | Swin-B/384 | EfficientNetV2-M (IN-21k→1k) |
| 3 | `vit_base_384__abl_HC__f0` | ViT-B/16-384 (IN-21k→1k) | ConvNeXt-B/384 |
| 4 | `vit_base_384__abl_HE__f0` | ViT-B/16-384 | EfficientNetV2-M |

### Kiến trúc
Mỗi ảnh đi qua **cả hai** backbone (mỗi backbone dùng chung cho 2 ảnh). Mỗi backbone cho **một token riêng** → 6 token vào
gate của MoE-D, gate tự cân CNN (kết cấu/bề mặt cục bộ) với ViT (ngữ cảnh toàn ảnh) cho từng lesion:
```
clin ─┬─ ViT ─ token clin       ┐
      └─ CNN ─ token clin_cnn   │
derm ─┬─ ViT ─ token derm       ├─ expert theo nguồn + gate (6) ─ 3 head long-tail (τ 0 / 0.5 / 1) ─ p
      └─ CNN ─ token derm_cnn   │
demo ─ MLP ─ token demo         │
MONET ─ MLP ─ token monet       ┘
```
Giữ nguyên mọi thành phần của M0 (modality dropout 0.15, head phụ từng token, sampler q0.5). Backbone CNN nhận cùng lr,
drop path và grad checkpointing với backbone ViT; ảnh được chuẩn hoá lại cho nhánh CNN nếu mean/std khác
(ViT augreg và EfficientNetV2 bản TF dùng 0.5/0.5, Swin và ConvNeXt dùng ImageNet). Cột `gate_*` trong `oof.csv` cho
biết mỗi lesion dựa vào CNN hay ViT bao nhiêu (bảng cuối notebook).

**Thời gian (ước lượng, A100 80GB):** 2 backbone 384 px → ~2× M0, khoảng 1.5–2 giờ/run; 2 run song song → **~4 giờ**.
Số đo thật in ra sau run đầu tiên. Hai máy: máy thứ 2 đặt `HUONG = "down"`.

**Trước khi chạy:** Runtime → **A100 GPU**, bật High-RAM → **Run all**. Ngắt giữa chừng: mở lại, Run all — run đã xong
(trên Drive) được bỏ qua. Không cần HF_TOKEN (cả 4 backbone tải từ timm).

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu MILK10k
DRIVE_DIR = "/content/drive/MyDrive/milk10k_stage10"          # thư mục riêng của stage 10
M0_DIR = "/content/drive/MyDrive/milk10k_ablation/outputs"    # run M0 (stage 6) để so sánh — chỉ đọc

# ===================== CHỌN THÍ NGHIỆM =====================
EXPERIMENT = "experiments/stage10_hybrid.yaml"
BACKBONES = ["swin_base_384", "vit_base_384"]   # nhánh ViT
VARIANTS = ["HC", "HE"]                         # nhánh CNN: HC = ConvNeXt-B · HE = EfficientNetV2-M
FOLDS = [0]                                     # 4 run fold 0 | cấu hình thắng chạy 5 fold: [0, 1, 2, 3, 4]

HUONG = "top"                       # top: từ trên xuống | down: từ dưới lên (máy thứ 2)
TIME_BUDGET_H = 23.0                # không bắt đầu run mới nếu không kịp xong trước mốc này
JOBS_PER_GPU = "auto"               # auto: 2 nếu VRAM >= 70GB, 1 nếu ít hơn | số cụ thể (giảm nếu CUDA out of memory)

In [ ]:
# ===================== THAM SỐ TRAIN (recipe M0) =====================
# Hai backbone 384 px trong một mô hình -> batch 8 x accum 8 (hiệu dụng 64 như M0) + grad checkpointing.
PARAMS = {
    "train.batch_size": 8,              # 8 lesion = 16 ảnh x 2 backbone mỗi bước
    "train.accum_steps": 8,             # batch hiệu dụng 64
    "model.grad_checkpointing": True,   # tắt (False) nếu VRAM còn dư để train nhanh hơn
    "train.amp_dtype": "bf16",
    "train.epochs": 20,
    "train.patience": 5,
    "seed": 42,
    "predict.tta": True,
}
RUN_SUFFIX = ""

## 1. Google Drive, GPU và số run song song

In [ ]:
import os, subprocess, torch
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR + "/outputs", exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime -> Change runtime type -> A100 GPU"
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 2**30
if "A100" not in props.name:
    print(f"CẢNH BÁO: đang dùng {props.name}, không phải A100. bf16 cần GPU Ampere trở lên (A100, L4, H100).")
jobs = JOBS_PER_GPU if JOBS_PER_GPU != "auto" else (2 if vram >= 70 else 1)
cpus = os.cpu_count()
PARAMS["data.num_workers"] = max(2, cpus // jobs - 1)
GPUS = ",".join(["0"] * jobs)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # giảm phân mảnh VRAM
print(f"{props.name} {vram:.0f}GB | {cpus} CPU | {jobs} run song song | {PARAMS['data.num_workers']} worker/run")

## 2. Code, dữ liệu, token HuggingFace

In [ ]:
REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/content/MILK10k"            # ổ SSD của Colab: đọc ảnh nhanh hơn Drive
os.environ["MILK10K_ROOT"] = DATA_DIR
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "timm>=1.0" gdown openpyxl safetensors huggingface_hub
!git log -1 --format="Commit: %h %s"
!python scripts/setup_data.py --drive-url "{DRIVE_URL}" --dest {DATA_DIR}

In [ ]:
# Kiểm tra nhanh mô hình hybrid + lưới 4 run (~2 phút, không tải trọng số)
!python -m pytest tests/test_smoke.py -q -p no:cacheprovider -k "hybrid or stage10 or moe"

## 3. Đồng bộ kết quả lên Drive và danh sách run

In [ ]:
# Chạy tiếp: run đã xong trên Drive được chép về outputs/ và bỏ qua. --shared-dir: 2 máy không chạy trùng run.
restore_arg = (f"--restore {DRIVE_DIR}/outputs --shared-dir {DRIVE_DIR}/outputs"
               + (" --reverse" if HUONG == "down" else ""))
GPU_LOG = f"outputs/_gpu_usage_{HUONG}.csv"
os.makedirs("outputs", exist_ok=True)
# Đồng bộ nền mỗi 5 phút (bỏ checkpoint .pt) + ghi mức dùng GPU mỗi phút
sync_cmd = f"while true; do rsync -a --exclude '*.pt' outputs/ '{DRIVE_DIR}/outputs/'; sleep 300; done"
if "sync_proc" not in globals() or sync_proc.poll() is not None:
    sync_proc = subprocess.Popen(["bash", "-c", sync_cmd])
    gpu_log = subprocess.Popen("nvidia-smi --query-gpu=timestamp,utilization.gpu,memory.used,power.draw "
                               f"--format=csv -l 60 > {GPU_LOG}", shell=True)
print("Đồng bộ nền:", DRIVE_DIR + "/outputs")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
select_arg = (f"--backbones {','.join(BACKBONES)} --variants {','.join(VARIANTS)}"
              + (f" --folds {','.join(map(str, FOLDS))}" if FOLDS else ""))
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/colab_a100.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --dry-run

## 4. Train

In [ ]:
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/colab_a100.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --gpus {GPUS} --est-run-h 1.8 --time-budget-h {TIME_BUDGET_H}

## 5. Kết quả: so với M0 fold 0 + trọng số gate CNN / ViT
Đọc `metrics.json` và `oof.csv` của từng run (val fold 0). `gate CNN` = tổng trọng số gate của 2 token CNN, trung bình trên
lesion val (0 = bỏ qua CNN, 1 = chỉ dùng CNN). File nộp test của từng run: `outputs/<run>/submission.csv`
(**chưa nộp** — chọn cấu hình theo val trước, chạy 5 fold rồi mới nộp 1 lần).

In [ ]:
import json, glob
import pandas as pd
from IPython.display import display
!rsync -a --exclude '*.pt' outputs/ "{DRIVE_DIR}/outputs/"
CLASSES = ["AKIEC", "BCC", "BEN_OTH", "BKL", "DF", "INF", "MAL_OTH", "MEL", "NV", "SCCKA", "VASC"]
dirs = sorted(glob.glob(f"{DRIVE_DIR}/outputs/*__abl_H[CE]__f*")) + sorted(glob.glob(f"{M0_DIR}/swin_base_384__abl_M0__f0"))
rows = []
for d in dirs:
    try:
        m = json.load(open(f"{d}/metrics.json")); oof = pd.read_csv(f"{d}/oof.csv")
    except FileNotFoundError:
        continue
    o = m["overall"]
    r = {"run": d.rstrip("/").split("/")[-1], "Dice val": o["dice"], "AUC": o["auc"],
         "Sens": o.get("sensitivity"), "BMA": o.get("balanced_acc")}
    g = oof.filter(like="gate_")
    if len(g.columns):
        cnn = [c for c in g.columns if c.endswith("_cnn")]
        r["gate CNN"] = g[cnn].sum(1).mean() if cnn else 0.0
        r.update({c.removeprefix("gate_"): g[c].mean() for c in g.columns})
    rows.append(r)
tab = pd.DataFrame(rows).set_index("run")
if "swin_base_384__abl_M0__f0" in tab.index:
    tab["Δ Dice so với M0"] = tab["Dice val"] - tab.loc["swin_base_384__abl_M0__f0", "Dice val"]
display(tab.round(3))
tab.round(4).to_csv(f"{DRIVE_DIR}/outputs/stage10_summary.csv")
print("Đã lưu:", f"{DRIVE_DIR}/outputs/stage10_summary.csv")

In [ ]:
# Dice val theo lớp (fold 0) — lớp nào được lợi từ nhánh CNN
per = {}
for d in dirs:
    try:
        oof = pd.read_csv(f"{d}/oof.csv")
    except FileNotFoundError:
        continue
    p = oof[CLASSES].to_numpy(); y = oof["label"].to_numpy(); pred = p.argmax(1)
    per[d.rstrip("/").split("/")[-1]] = {c: 2 * ((y == k) & (pred == k)).sum() / max((y == k).sum() + (pred == k).sum(), 1)
                                         for k, c in enumerate(CLASSES)}
display(pd.DataFrame(per).T.round(3))

In [ ]:
# Mức dùng GPU trong lúc train (lấy mẫu mỗi phút)
import pandas as pd
u = pd.read_csv(GPU_LOG, skipinitialspace=True)
util = u.filter(like="utilization").iloc[:, 0].str.rstrip(" %").astype(float)
mem = u.filter(like="memory.used").iloc[:, 0].str.rstrip(" MiB").astype(float) / 1024
print(f"GPU util trung bình {util.mean():.0f}% (trung vị {util.median():.0f}%), VRAM tối đa {mem.max():.1f}GB, {len(u)} mẫu")